In [27]:
import csv
import pathlib
import numpy as np

from collections import defaultdict
from tqdm import tqdm

1. Split up DO and PO into separate result sets
2. Define coding scheme "anchors" -- fixed values which will be mapped onto the binary code. 
   P.S. The fact that they're fixed does not matter since we will have all possible codes, so every code and its complement will be included.
3. Generate all possible codes
4. Generate versions of the results w/ coding scheme for both DO and PO
   Label for actual HAAP codes
5. save in a gitignored directory (256 x 2 files)

In [28]:
(np.arange(2**8 // 2, dtype=np.uint8)[:, None] >> np.arange(7, -1, -1)) & 1

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 1],
       [0, 0, 0, ..., 0, 1, 0],
       ...,
       [0, 1, 1, ..., 1, 0, 1],
       [0, 1, 1, ..., 1, 1, 0],
       [0, 1, 1, ..., 1, 1, 1]])

In [29]:
# all_codes = (np.arange(2**8, dtype=np.uint8)[:, None] >> np.arange(7, -1, -1)) & 1
all_codes = (np.arange(2**8 // 2, dtype=np.uint8)[:, None] >> np.arange(7, -1, -1)) & 1

HAAP_DO = np.array([0,0,0,0,1,1,1,1])
HAAP_PO = np.array([1,0,1,1,0,1,0,0])

FIXED_ANCHOR = ("pronoun", "animate", "definite", "given", "pronoun", "animate", "definite", "given")

In [30]:
HAAP_PO, HAAP_DO

(array([1, 0, 1, 1, 0, 1, 0, 0]), array([0, 0, 0, 0, 1, 1, 1, 1]))

In [68]:
samsies[78]

(78, 65)

In [78]:
all_codes[124]

array([0, 1, 1, 1, 1, 1, 0, 0])

In [32]:
np.count_nonzero(HAAP_PO!=(1-all_codes[75]))

0

In [33]:
hamming = {
    "do": defaultdict(int),
    "po": defaultdict(int)
}

for i, code in enumerate(all_codes):
    # do diff
    hamming['do'][i] = np.count_nonzero(HAAP_DO!=code)
    hamming['po'][i] = np.count_nonzero(HAAP_PO!=code)

In [34]:
all_codes[15]

array([0, 0, 0, 0, 1, 1, 1, 1])

In [35]:
1-all_codes[65]

array([1, 0, 1, 1, 1, 1, 1, 0])

In [36]:
1-all_codes[73]

array([1, 0, 1, 1, 0, 1, 1, 0])

In [37]:
all_codes[75]

array([0, 1, 0, 0, 1, 0, 1, 1])

In [38]:
(1-all_codes[75])

array([1, 0, 1, 1, 0, 1, 0, 0])

In [39]:
all_codes[78] == all_codes[68]

array([ True,  True,  True,  True, False,  True, False,  True])

In [40]:
all_codes[78]

array([0, 1, 0, 0, 1, 1, 1, 0])

In [41]:
all_codes[68][:4] == (1-HAAP_PO[:4])

array([ True,  True,  True,  True])

In [42]:
all_codes3d = all_codes[:, np.newaxis, :]

diffs = np.abs(all_codes3d - all_codes)

In [43]:
all_codes[0], all_codes[127]

(array([0, 0, 0, 0, 0, 0, 0, 0]), array([0, 1, 1, 1, 1, 1, 1, 1]))

In [44]:
diffs.sum(2)

array([[0, 1, 1, ..., 6, 6, 7],
       [1, 0, 2, ..., 5, 7, 6],
       [1, 2, 0, ..., 7, 5, 6],
       ...,
       [6, 5, 7, ..., 0, 2, 1],
       [6, 7, 5, ..., 2, 0, 1],
       [7, 6, 6, ..., 1, 1, 0]])

In [45]:
np.where(diffs.sum(2) == 8)

(array([], dtype=int64), array([], dtype=int64))

In [46]:
def feature2code(feature,code):
    vector = [0,0,0,0,0,0,0,0]
    for i, (f, fa, c) in enumerate(zip(feature, FIXED_ANCHOR, code)):
        if f == fa:
            vector[i] = c
        else:
            vector[i] = 1-c
    return vector

In [47]:
feature2code(("pronoun", "inanimate", "definite", "given", "noun", "animate", "indefinite", "new"), HAAP_PO)

[1, 1, 1, 1, 1, 1, 1, 1]

In [48]:
feature2code(("pronoun", "animate", "definite", "given", "pronoun", "animate", "definite", "given"), HAAP_PO)

[1, 0, 1, 1, 0, 1, 0, 0]

In [49]:
# FIXED_ANCHORS = {
#     "do": ("noun", "inanimate", "indefinite", "new", "pronoun", "animate", "definite", "given"),
#     "po": ("pronoun", "inanimate", "definite", "given", "noun", "animate", "indefinite", "new"),
# }

# def feature2code(feature,code,dative):
#     vector = [0,0,0,0,0,0,0,0]
#     for i, (f, fa, c) in enumerate(zip(feature, FIXED_ANCHORS[dative], code)):
#         if f == fa:
#             vector[i] = c
#         else:
#             vector[i] = 1-c
#     return vector

In [50]:
# feature2code(("pronoun", "inanimate", "definite", "given", "noun", "animate", "indefinite", "new"), [1,1,1,1,1,1,1,1], "do")

In [51]:
def read_csv_dict(path):
    data = []
    with open(path, "r") as f:
        reader = csv.DictReader(f)
        for line in reader:
            data.append(line)
    return data

def write_csv_dict(data, path):
    with open(path, "w") as f:
        writer = csv.DictWriter(f, fieldnames=data[0].keys())
        writer.writeheader()
        for line in data:
            writer.writerow(line)

In [52]:
# results = read_csv_dict("../data/results/simulation-results/final-results-25-10-16.csv")
adaptation = read_csv_dict("../data/experiments/final-adaptation-25-10-16.csv")

In [53]:
def get_features(entry):
    return entry['theme_pronominality'], entry['theme_animacy'], entry['theme_definiteness'], entry['theme_givenness'], entry['recipient_pronominality'], entry['recipient_animacy'], entry['recipient_definiteness'], entry['recipient_givenness'], int(entry['length_diff'])

In [54]:
idxes = set()
for entry in adaptation:
    idxes.add(entry['idx'])

In [55]:
len(idxes)

12096

In [56]:
all_codes[65]

array([0, 1, 0, 0, 0, 0, 0, 1])

In [57]:
'''
id, seed, givennesstemplate, dative, do, pp, [vector], lengthdiff
'''

dative_results = defaultdict(lambda: defaultdict(list))

for entry in tqdm(adaptation):
    dative = entry['dative']
    features = get_features(entry)
    for code_id, code in enumerate(all_codes):
        haaps = {
            'do': False,
            'po': False,
            'do_recipient': False,
            'po_recipient': False,
            'do_theme': False,
            'po_theme': False,
        }
        haap_multiplier = 1
        haap_theme_multiplier = 1
        haap_recipient_multiplier = 1
        vector = feature2code(features[:-1], code)
        tp,ta,td,tg,rp,ra,rd,rg = vector
        if sum(code==HAAP_DO) == 8 or sum(code == (1-HAAP_DO)) == 8:
            haaps['do'] = True
            haap_multiplier = 1
            if sum(code == (1-HAAP_DO)) == 8:
                haap_multiplier = -1
        if sum(code==HAAP_PO) == 8 or sum(code==(1-HAAP_PO)) == 8:
            haaps['po'] = True
            haap_multiplier = 1
            if sum(code == (1-HAAP_PO)) == 8:
                haap_multiplier = -1
        if sum(code[:4]==HAAP_DO[:4]) == 4 or sum(code[:4]==(1-HAAP_DO[:4])) == 4:
            haaps['do_theme'] = True
            haap_theme_multiplier = 1
            if sum(code[:4]==(1-HAAP_DO[:4])) == 4:
                haap_theme_multiplier = -1
        if sum(code[:4]==HAAP_PO[:4]) == 4 or sum(code[:4]==(1-HAAP_PO[:4])) == 4:
            haaps['po_theme'] = True
            haap_theme_multiplier = 1
            if sum(code[:4]==(1-HAAP_PO[:4])) == 4:
                haap_theme_multiplier = -1
        if sum(code[4:]==HAAP_DO[4:]) == 4 or sum(code[4:]==(1-HAAP_DO[4:])) == 4:
            haaps['do_recipient'] = True
            haap_recipient_multiplier = 1
            if sum(code[4:]==(1-HAAP_DO[4:])) == 4:
                haap_recipient_multiplier = -1
        if sum(code[4:]==HAAP_PO[4:]) == 4 or sum(code[4:]==(1-HAAP_PO[4:])) == 4:
            haaps['po_recipient'] = True
            haap_recipient_multiplier = 1
            if sum(code[4:]==(1-HAAP_PO[4:])) == 4:
                haap_recipient_multiplier = -1
            
        dative_results[dative][code_id].append({
            'idx': entry['idx'],
            'item': entry['item'],
            'dative': dative,
            'length_diff': entry['length_diff'],
            "code_id": code_id,
            'code_score': sum(vector),
            'code_score_theme': sum(vector[:4]),
            'code_score_recipient': sum(vector[4:]),
            'haap_multiplier': haap_multiplier,
            'haap_theme_multiplier': haap_theme_multiplier,
            'haap_recipient_multiplier': haap_recipient_multiplier,
            'haap_do': haaps['do'],
            'haap_po': haaps['po'],
            'haap_do_theme': haaps['do_theme'],
            'haap_po_theme': haaps['po_theme'],
            'haap_do_recipient': haaps['do_recipient'],
            'haap_po_recipient': haaps['po_recipient'],
            'hamming_do': hamming['do'][code_id],
            'hamming_po': hamming['po'][code_id],
        })
            
#         if dative == 'do':
#             if sum(code == HAAP_DO) == 8:
#                 haap = True
#             else:
#                 haap = False          
                
#         elif dative == 'po':
#             if sum(code == HAAP_PO) == 8:
#                 haap = True
#             else:
#                 haap = False
                
        

100%|█████████████████████████████████████████████████████████████████████████| 12096/12096 [00:27<00:00, 441.28it/s]


In [58]:
def decompose(code):
    return code[:4], code[4:]

In [59]:
'''
query all
decompose into r and t
if r == all true or r = all false and t == all true or t == all false:
samsies.append()
'''

samsies = []
for i, code1 in enumerate(all_codes):
    for j, code2 in enumerate(all_codes):
        theme1, recipient1 = decompose(code1)
        theme2, recipient2 = decompose(code2)
        theme_xor = (theme1 ^ theme2).mean()
        recip_xor = (recipient1 ^ recipient2).mean()
        if i != j:
            if (theme_xor == 0 or theme_xor == 1) and (recip_xor == 0 or recip_xor == 1):
                samsies.append((i,j))

unique_samsies = set(map(lambda x: tuple(sorted(x)), samsies))

samsies_map = {k:v for k,v in samsies}

In [60]:
redundant = [{"code_id": i} for i,j in unique_samsies]

In [61]:
pathlib.Path("../data/results/simulation-results/haap-25-10-18/").mkdir(parents=True, exist_ok=True)

In [62]:
write_csv_dict(redundant, '../data/results/simulation-results/redundant-haaps.csv')

In [63]:
for dative, code_results in dative_results.items():
    for code_id, res in code_results.items():
        write_csv_dict(res, f"../data/results/simulation-results/haap-25-10-18/{dative}_{code_id}.csv")

In [148]:
# ("pronoun", "animate", "definite", "given", "pronoun", "animate", "definite", "given")
all_codes[76]

array([0, 1, 0, 0, 1, 1, 0, 0])

In [144]:
samsies_map[75]

68

In [125]:
HAAP_PO

array([1, 0, 1, 1, 0, 1, 0, 0])

In [129]:
all_codes[67]

array([0, 1, 0, 0, 0, 0, 1, 1])

In [16]:
all_codes[75], HAAP_PO

(array([0, 1, 0, 0, 1, 0, 1, 1]), array([1, 0, 1, 1, 0, 1, 0, 0]))

In [139]:
all_codes[127]

array([0, 1, 1, 1, 1, 1, 1, 1])

In [134]:
HAAP_DO

array([0, 0, 0, 0, 1, 1, 1, 1])

In [136]:
all_codes[78]

array([0, 1, 0, 0, 1, 1, 1, 0])

In [18]:
all_codes[127], all_codes[15]

(array([0, 1, 1, 1, 1, 1, 1, 1]), array([0, 0, 0, 0, 1, 1, 1, 1]))

In [18]:
all_codes[65]

array([0, 1, 0, 0, 0, 0, 0, 1])

In [17]:
all_codes[177], all_codes[190], all_codes[65], all_codes[78]

IndexError: index 177 is out of bounds for axis 0 with size 128

In [40]:
177-78, 190-65

(99, 125)

In [33]:
all_codes[0], all_codes[112]

(array([0, 0, 0, 0, 0, 0, 0, 0]), array([0, 1, 1, 1, 0, 0, 0, 0]))

In [36]:
180-112

68

In [37]:
all_codes[180], all_codes[68]

(array([1, 0, 1, 1, 0, 1, 0, 0]), array([0, 1, 0, 0, 0, 1, 0, 0]))

In [29]:
177-65, 190-78

(112, 112)

In [22]:
vector

[1, 1, 1, 0, 0, 1, 1, 0]

In [9]:
get_features(results[0])[:-1]

feature2code(get_features(results[0])[:-1], HAAP_DO)

[0, 0, 0, 1, 1, 1, 0, 0]

In [51]:
def haap_coding(entry):
    dative = entry['dative']
    features = get_features(entry)[:-1]
    tp,ta,td,tg,rp,ra,rd,rg = [1,1,1,1,0,0,0,0]
    
    # animacy is globally coded:
    if features[1] == "animate":
        ta = 0
    if features[5] == "animate":
        ra = 1
    # exposure-wise harmonic alignment coding
    if dative == "do":
        if features[0] == "pronoun":
            tp = 0
        if features[2] == "definite":
            td = 0
        if features[3] == "given":
            tg = 0
        if features[4] == "pronoun":
            rp = 1
        if features[6] == "definite":
            rd = 1
        if features[7] == "given":
            rg = 1
            
    elif dative == "pp":
        if features[0] == "noun":
            tp = 0
        if features[2] == "indefinite":
            td = 0
        if features[3] == "new":
            tg = 0
        if features[4] == "noun":
            rp = 1
        if features[6] == "indefinite":
            rd = 1
        if features[7] == "new":
            rg = 1
    return [tp,ta,td,tg,rp,ra,rd,rg]

In [23]:
2**16

65536

In [ ]:
'''
Scheme design:

assign binary code based on fixed positional values:

do: (pronoun, animate, definite, given, pronoun, animate, definite, given)
po: (pronoun, animate, definite, given, pronoun, animate, definite, given)

HAAP:
do: (0,0,0,0,1,1,1,1)
po: (1,0,1,1,0,1,0,0)

or in the full vector form:

'''

In [17]:
def code(entry, scheme):
    '''codes an entry based on the specified'''

[0, 0, 0, 1, 1, 1, 0, 0]

In [18]:
get_features(results[0])

('pronoun',
 'animate',
 'definite',
 'new',
 'pronoun',
 'animate',
 'indefinite',
 'new',
 0)

'\n1. Split up DO and PO into separate result sets\n2. Define coding scheme "anchors" -- fixed values which will be mapped onto the binary code. \n   P.S. The fact that they\'re fixed does not matter since we will have all possible codes, so every code and its complement will be included.\n3. Generate all possible codes\n4. Generate versions of the results w/ coding scheme for both DO and PO\n   Label for actual HAAP codes\n5. save in a gitignored directory (256 x 2 files)\n'